# Results Analysis — Multi-Tool Agent Harness for RAN Energy Optimization

Compares four experimental conditions:
- **Baseline A** — Rule-based (no LLM)
- **Baseline B** — Open-loop LLM (no tools, no validation)
- **Baseline C** — Digital twin only (Planner + Validator + VIAVI, no additional tools)
- **Full harness** — Baseline C + all 5 tools

Metrics: energy savings, QoS violation rate, decision quality, per-tool ablation.

**Demo mode**: when no `output/` results exist, synthetic plausible data is generated so all cells run end-to-end.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# ── paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
# CML runs write to a sibling 'RAN ES Harness/output' directory when downloaded
# locally; fall back to the in-repo output/ for runs executed from this machine.
_candidate  = PROJECT_ROOT.parent / 'RAN ES Harness' / 'output'
OUTPUT_DIR  = _candidate if _candidate.exists() else PROJECT_ROOT / 'output'
print(f'OUTPUT_DIR → {OUTPUT_DIR}')

# ── plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':       150,
    'font.family':      'sans-serif',
    'font.size':        11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

CONDITION_ORDER  = ['baseline_rules', 'baseline_openloop', 'baseline_digital_twin', 'full_harness']
CONDITION_LABELS = {
    'baseline_rules':          'A: Rule-based',
    'baseline_openloop':       'B: Open-loop LLM',
    'baseline_digital_twin':   'C: Digital twin',
    'full_harness':            'Proposed: Full harness',
}
PALETTE = {
    'baseline_rules':          '#9e9e9e',
    'baseline_openloop':       '#ef9a9a',
    'baseline_digital_twin':   '#90caf9',
    'full_harness':            '#1565c0',
}
TOOL_NAMES = ['historical_kpi', 'traffic_forecast', 'alarm_fault', 'interference', 'energy_pricing']

print('✓ Imports OK')

## 1. Load results

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_all_results() -> pd.DataFrame:
    """Load all iterations.jsonl files from output/ and return a flat DataFrame."""
    records = []
    for run_dir in sorted(OUTPUT_DIR.glob('run_*')):
        jsonl = run_dir / 'iterations.jsonl'
        if not jsonl.exists():
            continue
        rows = load_jsonl(jsonl)
        for r in rows:
            r['run_dir'] = run_dir.name
            records.append(r)
    return pd.DataFrame(records)


df_raw = load_all_results()

if df_raw.empty:
    print('⚠  No results found in output/ — switching to DEMO MODE with synthetic data.')
    DEMO_MODE = True
else:
    DEMO_MODE = False
    print(f'✓ Loaded {len(df_raw):,} iteration records from {df_raw["run_dir"].nunique()} runs')
    print(df_raw['condition'].value_counts().to_string())

In [ ]:
# ── Demo mode: generate plausible synthetic results ───────────────────────────
if DEMO_MODE:
    rng = np.random.default_rng(42)
    N   = 30  # iterations per condition

    def _make_condition(condition, n_cells=6, sleep_mean=0.25, sleep_std=0.10,
                        qos_viol_prob=0.15, approval_rate=1.0, proposed_mean=2.5):
        rows = []
        for i in range(N):
            n_proposed = max(0, int(rng.normal(proposed_mean, 0.8)))
            n_approved = int(n_proposed * approval_rate)
            n_rejected = n_proposed - n_approved
            sleep_frac = np.clip(rng.normal(sleep_mean, sleep_std), 0, 1)
            rows.append({
                'iteration':          i + 1,
                'condition':          condition,
                'n_proposed':         n_proposed,
                'n_approved':         n_approved,
                'n_rejected':         n_rejected,
                'sleep_fraction':     round(sleep_frac, 3),
                'qos_violated':       int(rng.random() < qos_viol_prob),
                'mean_throughput_mbps': round(rng.normal(7.2 - sleep_frac * 1.5, 0.4), 2),
                'planner_elapsed_s':  round(rng.uniform(3, 8), 2),
                'validator_elapsed_s':round(rng.uniform(4, 10), 2) if approval_rate < 1 else 0.0,
                'total_elapsed_s':    round(rng.uniform(8, 20), 2),
                'tools_used':         [],
            })
        return rows

    synthetic = (
        _make_condition('baseline_rules',        sleep_mean=0.22, qos_viol_prob=0.20, approval_rate=1.00, proposed_mean=1.8) +
        _make_condition('baseline_openloop',     sleep_mean=0.35, qos_viol_prob=0.30, approval_rate=1.00, proposed_mean=3.2) +
        _make_condition('baseline_digital_twin', sleep_mean=0.30, qos_viol_prob=0.10, approval_rate=0.75, proposed_mean=3.0) +
        _make_condition('full_harness',          sleep_mean=0.42, qos_viol_prob=0.03, approval_rate=0.88, proposed_mean=3.5)
    )

    # Ablation runs — each tool added one at a time on top of baseline C
    ablation_params = {
        'ablation_historical_kpi':   dict(sleep_mean=0.33, qos_viol_prob=0.09),
        'ablation_traffic_forecast': dict(sleep_mean=0.34, qos_viol_prob=0.08),
        'ablation_alarm_fault':      dict(sleep_mean=0.31, qos_viol_prob=0.06),
        'ablation_interference':     dict(sleep_mean=0.32, qos_viol_prob=0.07),
        'ablation_energy_pricing':   dict(sleep_mean=0.35, qos_viol_prob=0.08),
    }
    for label, params in ablation_params.items():
        synthetic += _make_condition(label, approval_rate=0.80, proposed_mean=3.1, **params)

    df_raw = pd.DataFrame(synthetic)
    print(f'✓ Demo data: {len(df_raw)} rows across {df_raw["condition"].nunique()} conditions')

In [ ]:
# ── Normalise columns that may be absent in real results ──────────────────────

# A QoS violation is flagged when any AWAKE cell reports avg_qos below this
# threshold. Aggressive sleep decisions push traffic onto the coverage layer,
# saturating it and degrading QoS for awake cells — the per-cell check catches
# this cascade where the network-level aggregate would not.
QOS_VIOLATION_THRESHOLD_MBPS = 5.0


def extract_sleep_fraction(row):
    """Derive sleep fraction from post_kpis if present, else use pre-computed column."""
    if 'sleep_fraction' in row and pd.notna(row['sleep_fraction']):
        return row['sleep_fraction']
    post = row.get('post_kpis', {})
    if isinstance(post, dict) and post:
        # VIAVI harness format: {"sleeping_cells": N, "total_cells": M, "avg_throughput_mbps": X}
        if 'sleeping_cells' in post and 'total_cells' in post:
            tc = post['total_cells']
            return post['sleeping_cells'] / tc if tc > 0 else 0.0
        # Legacy per-cell dict format
        sleeps = [v.get('sleep_state', 0) for v in post.values() if isinstance(v, dict)]
        if sleeps:
            return sum(sleeps) / len(sleeps)
    # Fallback: count sleep actions against total_cells (42 for the VIAVI 21-site scenario)
    approved = row.get('approved_actions', [])
    if isinstance(approved, list):
        n_sleep = sum(1 for a in approved if isinstance(a, dict) and a.get('action') == 'sleep')
        tc = (row.get('post_kpis') or {}).get('total_cells', 42)
        return n_sleep / tc if tc > 0 else 0.0
    return 0.0


def extract_qos_violated(row):
    """
    Flag an iteration as a QoS violation when any AWAKE cell reports
    avg_qos below QOS_VIOLATION_THRESHOLD_MBPS (and has active UEs).

    Rationale: aggressive sleep decisions push UE traffic onto the N12
    coverage layer. When N12 saturates, awake cells whose UEs are served
    by N12 experience degraded throughput — visible here as a low avg_qos
    on an n1_sleeping=False cell.  The network-level avg_throughput_mbps
    aggregate misses this effect because high-QoS cells mask the degraded ones.
    """
    if 'qos_violated' in row and pd.notna(row['qos_violated']):
        return int(row['qos_violated'])

    post = row.get('post_kpis', {})
    if not isinstance(post, dict) or not post:
        return 0

    per_site = post.get('per_site', {})
    if per_site:
        for site_data in per_site.values():
            if not isinstance(site_data, dict):
                continue
            # Only check awake cells with active UEs (avg_qos > 0)
            if not site_data.get('n1_sleeping', False):
                qos = site_data.get('avg_qos', 999.0)
                if 0 < qos < QOS_VIOLATION_THRESHOLD_MBPS:
                    return 1
        return 0

    # Fallback: network-level aggregate (less sensitive but always available)
    if 'avg_throughput_mbps' in post:
        return int(post['avg_throughput_mbps'] < QOS_VIOLATION_THRESHOLD_MBPS)

    return 0


df = df_raw.copy()
df['sleep_fraction']  = df.apply(extract_sleep_fraction, axis=1)
df['qos_violated']    = df.apply(extract_qos_violated, axis=1)
df['approval_rate']   = np.where(
    df['n_proposed'] > 0,
    df['n_approved'] / df['n_proposed'],
    np.nan
)
df['is_main_condition'] = df['condition'].isin(CONDITION_ORDER)

df_main = df[df['is_main_condition']].copy()
print('Conditions in dataset:', df['condition'].unique().tolist())
print(df_main.groupby('condition')[['sleep_fraction','qos_violated','approval_rate']].mean().round(3))

## 2. Energy savings

In [ ]:
# ── 2a. Mean sleeping fraction per condition ──────────────────────────────────
energy_stats = (
    df_main.groupby('condition')['sleep_fraction']
    .agg(['mean', 'std', 'count'])
    .reindex(CONDITION_ORDER)
    .reset_index()
)
energy_stats['se'] = energy_stats['std'] / np.sqrt(energy_stats['count'])

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    [CONDITION_LABELS[c] for c in energy_stats['condition']],
    energy_stats['mean'] * 100,
    yerr=energy_stats['se'] * 100,
    color=[PALETTE[c] for c in energy_stats['condition']],
    capsize=4, width=0.55, error_kw={'linewidth': 1.2},
)
ax.set_ylabel('Mean cells sleeping (%)')
ax.set_title('Energy Savings by Condition')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_energy_savings.pdf', bbox_inches='tight')
plt.show()
print(energy_stats[['condition','mean','se']].to_string(index=False))

In [ ]:
# ── 2b. Sleep fraction over iterations (line chart) ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
for cond in CONDITION_ORDER:
    sub = df_main[df_main['condition'] == cond].sort_values('iteration')
    ax.plot(sub['iteration'], sub['sleep_fraction'] * 100,
            label=CONDITION_LABELS[cond], color=PALETTE[cond],
            linewidth=1.8, alpha=0.85)

ax.set_xlabel('Iteration')
ax.set_ylabel('Cells sleeping (%)')
ax.set_title('Sleep Fraction Over Iterations')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_sleep_over_time.pdf', bbox_inches='tight')
plt.show()

## 3. QoS preservation

In [ ]:
# ── 3a. QoS violation rate per condition ──────────────────────────────────────
qos_stats = (
    df_main.groupby('condition')['qos_violated']
    .agg(['mean', 'sum', 'count'])
    .reindex(CONDITION_ORDER)
    .reset_index()
)
qos_stats.columns = ['condition', 'violation_rate', 'n_violations', 'n_iterations']

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    [CONDITION_LABELS[c] for c in qos_stats['condition']],
    qos_stats['violation_rate'] * 100,
    color=[PALETTE[c] for c in qos_stats['condition']],
    width=0.55,
)
ax.set_ylabel('QoS violation rate (%)')
ax.set_title('QoS Violation Rate by Condition')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_qos_violations.pdf', bbox_inches='tight')
plt.show()
print(qos_stats.to_string(index=False))

In [ ]:
# ── 3b. Energy savings vs QoS violation scatter (trade-off plot) ──────────────
fig, ax = plt.subplots(figsize=(6, 5))
for cond in CONDITION_ORDER:
    e = energy_stats.loc[energy_stats['condition'] == cond, 'mean'].values[0] * 100
    q = qos_stats.loc[qos_stats['condition'] == cond, 'violation_rate'].values[0] * 100
    ax.scatter(e, q, color=PALETTE[cond], s=120, zorder=3)
    ax.annotate(
        CONDITION_LABELS[cond], (e, q),
        textcoords='offset points', xytext=(6, 4), fontsize=9
    )

ax.set_xlabel('Mean cells sleeping (%)')
ax.set_ylabel('QoS violation rate (%)')
ax.set_title('Energy Savings vs QoS Preservation')
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.invert_yaxis()   # lower violation = better = higher on chart
ax.set_ylabel('QoS violation rate (%, lower is better ↑)')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_tradeoff.pdf', bbox_inches='tight')
plt.show()

## 4. Decision quality

In [ ]:
# ── Approval rate (only meaningful for LLM conditions with a validator) ───────
llm_conds = ['baseline_digital_twin', 'full_harness']
dec_stats = (
    df_main[df_main['condition'].isin(llm_conds) & df_main['n_proposed'].gt(0)]
    .groupby('condition')[['approval_rate', 'n_proposed', 'n_rejected']]
    .agg({'approval_rate': ['mean', 'std'], 'n_proposed': 'mean', 'n_rejected': 'mean'})
)
dec_stats.columns = ['approval_mean', 'approval_std', 'avg_proposed', 'avg_rejected']
print('Decision quality (LLM conditions with validator):')
print(dec_stats.round(3))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

# Approval rate
axes[0].bar(
    [CONDITION_LABELS[c] for c in llm_conds],
    dec_stats.loc[llm_conds, 'approval_mean'] * 100,
    yerr=dec_stats.loc[llm_conds, 'approval_std'] * 100,
    color=[PALETTE[c] for c in llm_conds], capsize=4, width=0.5
)
axes[0].set_ylabel('Action approval rate (%)')
axes[0].set_title('Validator Approval Rate')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())

# Avg proposed vs approved
x  = np.arange(len(llm_conds))
w  = 0.35
axes[1].bar(x - w/2, dec_stats.loc[llm_conds, 'avg_proposed'],
            width=w, color='#bdbdbd', label='Proposed')
axes[1].bar(x + w/2,
            dec_stats.loc[llm_conds, 'avg_proposed'] - dec_stats.loc[llm_conds, 'avg_rejected'],
            width=w, color=[PALETTE[c] for c in llm_conds], label='Approved')
axes[1].set_xticks(x)
axes[1].set_xticklabels([CONDITION_LABELS[c] for c in llm_conds], rotation=10, ha='right')
axes[1].set_ylabel('Actions per iteration (mean)')
axes[1].set_title('Proposed vs Approved Actions')
axes[1].legend(fontsize=9, frameon=False)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_decision_quality.pdf', bbox_inches='tight')
plt.show()

## 5. Ablation study

In [ ]:
# ── Ablation: each tool added to digital-twin baseline ───────────────────────
ABLATION_TOOL_MAP = {
    f'ablation_{t}': t for t in TOOL_NAMES
}

df_abl  = df[df['condition'].isin(list(ABLATION_TOOL_MAP.keys()) + ['baseline_digital_twin', 'full_harness'])].copy()
abl_agg = df_abl.groupby('condition')[['sleep_fraction', 'qos_violated']].mean().round(4)

abl_rows = []
if 'baseline_digital_twin' in abl_agg.index:
    base_energy = abl_agg.loc['baseline_digital_twin', 'sleep_fraction']
    base_qos    = abl_agg.loc['baseline_digital_twin', 'qos_violated']

    for cond_key, tool_name in ABLATION_TOOL_MAP.items():
        if cond_key not in abl_agg.index:
            continue
        row = abl_agg.loc[cond_key]
        abl_rows.append({
            'tool':               tool_name,
            'sleep_fraction':     row['sleep_fraction'],
            'delta_energy_pct':   (row['sleep_fraction'] - base_energy) * 100,
            'qos_violation_rate': row['qos_violated'],
            'delta_qos_pct':      (row['qos_violated']   - base_qos)   * 100,
        })

if not abl_rows:
    print('⚠  No ablation runs found — skipping ablation chart.')
    print('   Run full_harness.py with individual --tools flags to generate ablation data.')
    print('   Example: python experiments/full_harness.py --tools historical_kpi --iterations 96')
else:
    df_ablation = pd.DataFrame(abl_rows).set_index('tool')

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    colors_energy = ['#1565c0' if d >= 0 else '#ef5350' for d in df_ablation['delta_energy_pct']]
    axes[0].barh(df_ablation.index, df_ablation['delta_energy_pct'], color=colors_energy)
    axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
    axes[0].set_xlabel('Δ Energy savings vs Baseline C (pp)')
    axes[0].set_title('Marginal Energy Gain per Tool')

    colors_qos = ['#1565c0' if d <= 0 else '#ef5350' for d in df_ablation['delta_qos_pct']]
    axes[1].barh(df_ablation.index, df_ablation['delta_qos_pct'], color=colors_qos)
    axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
    axes[1].set_xlabel('Δ QoS violation rate vs Baseline C (pp)')
    axes[1].set_title('Marginal QoS Impact per Tool (negative = better)')

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_ablation.pdf', bbox_inches='tight')
    plt.show()
    print(df_ablation.round(4))

## 6. Tool correlation analysis
For full-harness runs: correlate per-iteration tool context signals with agent decisions.

In [ ]:
# ── Extract tool signals from logged tool_context if available ────────────────
df_fh = df[df['condition'] == 'full_harness'].copy()

has_context = 'tool_context' in df_fh.columns and df_fh['tool_context'].apply(
    lambda x: isinstance(x, dict) and bool(x)
).any()

if has_context:
    def _extract_signals(row):
        ctx    = row.get('tool_context', {}) or {}
        pricing = ctx.get('energy_pricing', {})
        faults  = ctx.get('alarm_fault', {})
        return pd.Series({
            'price':          pricing.get('current_price_per_kwh', np.nan),
            'is_peak':        int(pricing.get('current_tier', '') == 'peak'),
            'n_active_faults':sum(1 for v in faults.values() if isinstance(v, dict) and v.get('active')),
        })

    signals = df_fh.apply(_extract_signals, axis=1)
    df_corr = pd.concat([df_fh[['sleep_fraction', 'n_approved', 'qos_violated']], signals], axis=1).dropna()

    corr_matrix = df_corr.corr(method='spearman')
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
        center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5
    )
    ax.set_title('Spearman Correlation: Tool Signals vs Agent Decisions (Full Harness)')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'analysis' / 'fig_tool_correlation.pdf', bbox_inches='tight')
    plt.show()
else:
    print('Tool context not available in results — skipping correlation analysis.')
    print('(Run full_harness.py with VIAVI RSG to populate tool_context in iterations.jsonl)')

## 7. Statistical significance

In [ ]:
# ── Mann-Whitney U: full harness vs each baseline ────────────────────────────
fh_energy = df_main[df_main['condition'] == 'full_harness']['sleep_fraction'].values
fh_qos    = df_main[df_main['condition'] == 'full_harness']['qos_violated'].values

sig_rows = []
for cond in ['baseline_rules', 'baseline_openloop', 'baseline_digital_twin']:
    sub   = df_main[df_main['condition'] == cond]
    u_e, p_e = stats.mannwhitneyu(fh_energy, sub['sleep_fraction'].values, alternative='greater')
    u_q, p_q = stats.mannwhitneyu(fh_qos,    sub['qos_violated'].values,   alternative='less')
    sig_rows.append({
        'vs':                  CONDITION_LABELS[cond],
        'energy_U':            int(u_e),
        'energy_p':            round(p_e, 4),
        'energy_sig':          '✓' if p_e < 0.05 else '✗',
        'qos_U':               int(u_q),
        'qos_p':               round(p_q, 4),
        'qos_sig':             '✓' if p_q < 0.05 else '✗',
    })

df_sig = pd.DataFrame(sig_rows).set_index('vs')
print('Mann-Whitney U — Full Harness vs baselines (one-sided):')
print('  energy: full_harness > baseline (H_a)')
print('  qos:    full_harness < baseline violations (H_a)')
print()
print(df_sig.to_string())

## 8. Summary table (paper-ready)

In [ ]:
summary_rows = []
for cond in CONDITION_ORDER:
    sub = df_main[df_main['condition'] == cond]
    has_validator = cond in ['baseline_digital_twin', 'full_harness']
    summary_rows.append({
        'Condition':            CONDITION_LABELS[cond],
        'Mean sleep %':         f"{sub['sleep_fraction'].mean() * 100:.1f} ± {sub['sleep_fraction'].std() * 100:.1f}",
        'QoS viol. %':          f"{sub['qos_violated'].mean() * 100:.1f}",
        'Approval rate':        f"{sub['approval_rate'].mean() * 100:.1f}%" if has_validator else 'N/A',
        'Avg proposed':         f"{sub['n_proposed'].mean():.1f}",
        'Avg latency (s)':      f"{sub['total_elapsed_s'].mean():.1f}" if 'total_elapsed_s' in sub else 'N/A',
    })

df_summary = pd.DataFrame(summary_rows).set_index('Condition')
print('=== PAPER SUMMARY TABLE ===')
print(df_summary.to_string())

# Save as CSV for LaTeX import
df_summary.to_csv(PROJECT_ROOT / 'analysis' / 'table_summary.csv')
print('\n✓ Saved table_summary.csv  (import into LaTeX with \\input or pandas to_latex)')

In [ ]:
# ── LaTeX table snippet ───────────────────────────────────────────────────────
print('LaTeX table snippet:\n')
print(df_summary.to_latex(caption='Performance comparison across experimental conditions.',
                           label='tab:results', escape=False))